# Nivel I: Modelos Línea Base (SARIMAX, GLM Binomial Negativa y Gradient Boosting Poisson)

1. **Baseline 1A — SARIMAX independiente por cantón** sobre `log1p(casos)` (pronóstico 1 semana adelante mediante filtro de estado sin re-estimar parámetros en `val`/`test`).
2. **Baseline 1B — GLM Binomial Negativa global** implementando directamente la conclusión de sobredispersión.
3. **Baseline 1C — HistGradientBoosting con pérdida Poisson** referencia no lineal fuerte para conteos.

In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import ParameterSampler
from itertools import product
import mlflow

In [2]:
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")

ROOT_DIR = Path.cwd()
if ROOT_DIR.name == "notebooks":
    ROOT_DIR = ROOT_DIR.parent
os.chdir(ROOT_DIR)

df = pd.read_csv(ROOT_DIR / "data/processed/dengue_panel_clean.csv")
df = df.sort_values(["canton", "time_idx"]).reset_index(drop=True)
print("Panel limpio cargado:", df.shape)

## 1. Métricas unificadas de evaluación y conexión a MLflow

Usamos exactamente las mismas métricas en `val` (`2024`) y `test` (`2025`) para poder comparar Nivel I, Nivel II (ST-GNN) y Nivel III (ST-GNN + Kalman) en la misma tabla de MLflow.

In [3]:
def compute_count_metrics(y_true, y_pred, prefix=""):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.clip(np.asarray(y_pred, dtype=float), 0.0, None)

    mae = float(np.mean(np.abs(y_true - y_pred)))
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    rmsle = float(np.sqrt(np.mean((np.log1p(y_true) - np.log1p(y_pred)) ** 2)))
    denom = float(np.sum(np.abs(y_true)))
    wape = float(np.sum(np.abs(y_true - y_pred)) / denom) if denom > 0 else np.nan

    z_mask = y_true == 0
    p_mask = y_true > 0
    mae_zeros = float(np.mean(np.abs(y_true[z_mask] - y_pred[z_mask]))) if z_mask.any() else np.nan
    mae_pos = float(np.mean(np.abs(y_true[p_mask] - y_pred[p_mask]))) if p_mask.any() else np.nan

    p = f"{prefix}_" if prefix else ""
    return {
        f"{p}mae": round(mae, 4),
        f"{p}rmse": round(rmse, 4),
        f"{p}rmsle": round(rmsle, 4),
        f"{p}wape": round(wape, 4),
        f"{p}mae_zeros": round(mae_zeros, 4),
        f"{p}mae_positives": round(mae_pos, 4),
    }

mlruns_dir = (ROOT_DIR / "mlruns").resolve()
mlflow.set_tracking_uri(f"sqlite:///{(mlruns_dir / 'mlflow.db').as_posix()}")
mlflow.set_experiment("dengue_cr_spatiotemporal")

## 2. Ingeniería de rezagos explícita

- Usamos rezagos de `log1p(casos)` en `t-1, t-2, t-3, t-4, t-8` y la media móvil de 4 semanas.
- Incluimos rezagos biológicos de clima (`precip_total` en `t-3, t-4`, `temp_prom` en `t-1, t-4, t-8`, `nino34ssta` en `t-8`, `ndvi` en `t-4`).
- Mantenemos `week_sin`, `week_cos` y el atributo estático `urb`.

In [4]:
feat_df = df.copy()
feat_df["log_casos"] = np.log1p(feat_df["casos"])

for k in [1, 2, 3, 4, 8]:
    feat_df[f"log_casos_lag_{k}"] = feat_df.groupby("canton")["log_casos"].shift(k)

feat_df["log_casos_roll4"] = (
    feat_df.groupby("canton")["log_casos"]
    .transform(lambda s: s.shift(1).rolling(4).mean())
)

for col, lags in [("precip_total", [3, 4]), ("temp_prom", [1, 4, 8]), ("nino34ssta", [8]), ("ndvi", [4])]:
    for k in lags:
        feat_df[f"{col}_lag_{k}"] = feat_df.groupby("canton")[col].shift(k)

feature_cols = [
    "log_casos_lag_1", "log_casos_lag_2", "log_casos_lag_3", "log_casos_lag_4", "log_casos_lag_8",
    "log_casos_roll4",
    "precip_total_lag_3", "precip_total_lag_4",
    "temp_prom_lag_1", "temp_prom_lag_4", "temp_prom_lag_8",
    "nino34ssta_lag_8", "ndvi_lag_4",
    "week_sin", "week_cos", "urb",
]

model_df = feat_df.dropna(subset=feature_cols).reset_index(drop=True)
train_m = model_df[model_df["year"] <= 2023].copy()
val_m = model_df[model_df["year"] == 2024].copy()
test_m = model_df[model_df["year"] == 2025].copy()

print(f"Filas listas -> Train: {len(train_m):,} | Val: {len(val_m):,} | Test: {len(test_m):,}")

## 3. Modelo 1A: Auto-ARIMAX independiente por cantón (selección por AIC)

Como los 82 cantones tienen dinámicas muy distintas (desde 4% hasta 98% de ceros), en lugar de fijar un único orden `(2,0,1)` hacemos una **búsqueda Auto-ARIMA por cantón** sobre `(p, d, q)` con `p in [0,1,2,3]`, `d in [0,1]`, `q in [0,1,2]` usando `[week_sin, week_cos]` como exógenas **únicamente en `train` (`<= 2023`)** según el criterio **AIC**.

Luego aplicamos `.apply()` del mejor modelo sobre toda la serie para obtener pronósticos 1-semana-adelante en `val` y `test` sin re-estimar parámetros.

In [6]:
candidate_orders = list(product([0, 1, 2, 3], [0, 1], [0, 1, 2]))
sarimax_preds = pd.Series(index=df.index, dtype=float)
selected_orders = {}
failed_cantons = 0

for cid, grp in df.groupby("canton"):
    grp = grp.sort_values("time_idx")
    y_log = np.log1p(grp["casos"].to_numpy(dtype=float))
    exog = grp[["week_sin", "week_cos"]].to_numpy(dtype=float)
    tr_end = int((grp["year"] <= 2023).sum())

    best_aic = np.inf
    best_fit = None
    best_order = None

    for order in candidate_orders:
        try:
            fit_tr = SARIMAX(
                y_log[:tr_end],
                exog=exog[:tr_end],
                order=order,
                enforce_stationarity=False,
                enforce_invertibility=False,
            ).fit(disp=False, maxiter=80)
            if np.isfinite(fit_tr.aic) and fit_tr.aic < best_aic:
                best_aic = fit_tr.aic
                best_fit = fit_tr
                best_order = order
        except Exception:
            continue

    if best_fit is not None:
        res_full = best_fit.apply(y_log, exog=exog)
        pred_casos = np.clip(np.expm1(res_full.fittedvalues), 0.0, None)
        selected_orders[int(cid)] = str(best_order)
    else:
        failed_cantons += 1
        pred_casos = grp["casos"].shift(1).fillna(0.0).to_numpy(dtype=float)
        selected_orders[int(cid)] = "fallback_t1"

    sarimax_preds.loc[grp.index] = pred_casos

df["pred_sarimax"] = sarimax_preds
print("Top órdenes (p,d,q) seleccionados por AIC:")
print(pd.Series(selected_orders).value_counts().head(8))

val_sar = compute_count_metrics(df.loc[df["year"] == 2024, "casos"], df.loc[df["year"] == 2024, "pred_sarimax"], "val")
test_sar = compute_count_metrics(df.loc[df["year"] == 2025, "casos"], df.loc[df["year"] == 2025, "pred_sarimax"], "test")
print(f"Auto-ARIMAX (fallbacks={failed_cantons}/82) ->", val_sar, test_sar)

## 4. Modelo 1B: GLM Binomial Negativa global

Ajustamos una regresión **Binomial Negativa** con enlace logarítmico sobre `casos` en `train_m`.  
Como es un modelo global sobre los 82 cantones, incluye también la variable estática `urb` y la dinámica de rezagos.

In [7]:
X_tr_sm = sm.add_constant(train_m[feature_cols], has_constant="add")
X_va_sm = sm.add_constant(val_m[feature_cols], has_constant="add")
X_te_sm = sm.add_constant(test_m[feature_cols], has_constant="add")

glm_nb = sm.GLM(
    train_m["casos"],
    X_tr_sm,
    family=sm.families.NegativeBinomial(alpha=1.0),
).fit(maxiter=1000)

val_m["pred_glm_nb"] = np.clip(glm_nb.predict(X_va_sm), 0.0, None)
test_m["pred_glm_nb"] = np.clip(glm_nb.predict(X_te_sm), 0.0, None)

val_nb = compute_count_metrics(val_m["casos"], val_m["pred_glm_nb"], "val")
test_nb = compute_count_metrics(test_m["casos"], test_m["pred_glm_nb"], "test")
print("GLM NegBin ->", val_nb, test_nb)

## 5. Modelo 1C: HistGradientBoostingRegressor (Poisson)

Modelo no lineal basado en árboles con función de pérdida **Poisson** (apropiada para conteos no negativos) e incluyendo `canton` como variable categórica.

In [8]:
hgb_cols = feature_cols + ["canton"]
cat_idx = [hgb_cols.index("canton"), hgb_cols.index("urb")]

hgb = HistGradientBoostingRegressor(
    loss="poisson",
    max_iter=300,
    learning_rate=0.05,
    max_leaf_nodes=31,
    min_samples_leaf=30,
    categorical_features=cat_idx,
    random_state=42,
)
hgb.fit(train_m[hgb_cols], train_m["casos"])

val_m["pred_hgb_poisson"] = np.clip(hgb.predict(val_m[hgb_cols]), 0.0, None)
test_m["pred_hgb_poisson"] = np.clip(hgb.predict(test_m[hgb_cols]), 0.0, None)

val_hgb = compute_count_metrics(val_m["casos"], val_m["pred_hgb_poisson"], "val")
test_hgb = compute_count_metrics(test_m["casos"], test_m["pred_hgb_poisson"], "test")
print("HGB Poisson ->", val_hgb, test_hgb)

In [20]:
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
eval_m = pd.concat([val_m, test_m], ignore_index=True)

for ax, cid, name in zip(axes, [101, 305], ["Cantón 101", "Cantón 305"]):
    sub = eval_m[eval_m["canton"] == cid].sort_values("time_idx")
    sub_sar = df[(df["canton"] == cid) & (df["year"] >= 2024)].sort_values("time_idx")
    ax.plot(sub["time_idx"], sub["casos"], color="black", lw=1.8, label="Casos reales")
    ax.plot(sub_sar["time_idx"], sub_sar["pred_sarimax"], ls="--", alpha=0.85, label="Auto-ARIMAX")
    ax.plot(sub["time_idx"], sub["pred_glm_nb"], ls="-.", alpha=0.85, label="GLM NegBin")
    ax.plot(sub["time_idx"], sub["pred_hgb_poisson"], alpha=0.85, label="HGB Poisson")
    ax.axvline(int(test_m["time_idx"].min()), color="gray", ls=":", label="Inicio Test (2025)")
    ax.set_title(f"Pronóstico 1-semana adelante en Val (2024) y Test (2025) — {name}")
    ax.set_ylabel("Casos")
    ax.grid(False)
    ax.legend(loc="upper right", ncol=3)

axes[-1].set_xlabel("time_idx")
fig.tight_layout()
fig_b1 = ROOT_DIR / "data/processed/eda/08_nivel1_baselines_predicciones.png"
fig.savefig(fig_b1, dpi=150)
plt.show()

In [10]:
runs_info = [
    ("01_auto_arimax_per_canton", {"model_family": "auto_arimax_aic", "order_grid": "p0..3,d0..1,q0..2", "target_scale": "log1p"}, {**val_sar, **test_sar}),
    ("01_glm_negbin_global", {"model_family": "glm_negative_binomial", "alpha": 1.0, "n_features": len(feature_cols)}, {**val_nb, **test_nb}),
    ("01_hgb_poisson_global", {"model_family": "hist_gradient_boosting", "loss": "poisson", "max_iter": 300}, {**val_hgb, **test_hgb}),
]

for run_name, params, metrics in runs_info:
    with mlflow.start_run(run_name=run_name):
        mlflow.set_tags({"stage": "nivel1_baseline", "horizon": "1_week", "split_policy": "train<=2023|val=2024|test=2025"})
        mlflow.log_params(params)
        mlflow.log_metrics(metrics)
        mlflow.log_artifact(str(fig_b1), artifact_path="nivel1_plots")

exp_runs = mlflow.search_runs(experiment_names=["dengue_cr_spatiotemporal"])
cols_show = [c for c in ["tags.mlflow.runName", "tags.stage", "metrics.val_mae", "metrics.val_rmsle", "metrics.val_wape", "metrics.test_mae", "metrics.test_rmsle", "metrics.test_wape"] if c in exp_runs.columns]
display(exp_runs[cols_show].sort_values("metrics.val_rmsle", na_position="last"))

## 7. Fine-Tuning de HistGradientBoosting con Validación Cruzada Temporal (Expanding Window CV)

En la gráfica anterior vimos dos cosas clave:
1. **GLM NegBin** extrapola exponencialmente en el pico (`aprox 900` casos).
2. **HGB Poisson** se aplana cerca de `aprox 135–140` casos porque `min_samples_leaf=30` obliga a promediar demasiadas semanas de brote en las hojas superiores.

In [11]:
def make_expanding_time_folds(df_tr, n_splits=4, min_train_weeks=80):
    unique_t = np.sort(df_tr["time_idx"].unique())
    rem_weeks = len(unique_t) - min_train_weeks
    block = rem_weeks // n_splits
    folds = []
    for f in range(n_splits):
        val_start = unique_t[min_train_weeks + f * block]
        val_end = unique_t[min_train_weeks + (f + 1) * block] if f < n_splits - 1 else unique_t[-1] + 1
        tr_idx = df_tr.index[df_tr["time_idx"] < val_start].to_numpy()
        va_idx = df_tr.index[(df_tr["time_idx"] >= val_start) & (df_tr["time_idx"] < val_end)].to_numpy()
        folds.append((tr_idx, va_idx))
    return folds

folds = make_expanding_time_folds(train_m, n_splits=4, min_train_weeks=80)
for i, (tr_i, va_i) in enumerate(folds, 1):
    print(f"Fold {i}: train={len(tr_i):,} filas (t<={train_m.loc[tr_i, 'time_idx'].max()}) | val={len(va_i):,} filas (t={train_m.loc[va_i, 'time_idx'].min()}..{train_m.loc[va_i, 'time_idx'].max()})")

param_dist = {
    "target_mode": ["raw_poisson", "log1p_l2"],
    "learning_rate": [0.03, 0.05, 0.08],
    "max_iter": [250, 400],
    "max_leaf_nodes": [31, 63, 127],
    "min_samples_leaf": [5, 10, 20],
    "l2_regularization": [0.0, 0.5, 2.0],
}
candidates = list(ParameterSampler(param_dist, n_iter=16, random_state=42))

cv_results = []
with mlflow.start_run(run_name="01_hgb_tscv_tuning"):
    mlflow.set_tags({"stage": "nivel1_tuning", "cv_scheme": "expanding_window_4folds"})
    mlflow.log_params({"n_candidates": len(candidates), "n_folds": len(folds)})

    for idx, cfg in enumerate(candidates, 1):
        fold_rmsle, fold_mae = [], []
        loss_name = "poisson" if cfg["target_mode"] == "raw_poisson" else "squared_error"

        for tr_i, va_i in folds:
            X_f_tr, X_f_va = train_m.loc[tr_i, hgb_cols], train_m.loc[va_i, hgb_cols]
            y_f_va = train_m.loc[va_i, "casos"].to_numpy(dtype=float)
            y_f_tr = train_m.loc[tr_i, "casos"].to_numpy(dtype=float)
            if cfg["target_mode"] == "log1p_l2":
                y_f_tr = np.log1p(y_f_tr)

            est = HistGradientBoostingRegressor(
                loss=loss_name,
                learning_rate=cfg["learning_rate"],
                max_iter=cfg["max_iter"],
                max_leaf_nodes=cfg["max_leaf_nodes"],
                min_samples_leaf=cfg["min_samples_leaf"],
                l2_regularization=cfg["l2_regularization"],
                categorical_features=cat_idx,
                random_state=42,
            )
            est.fit(X_f_tr, y_f_tr)
            preds_va = est.predict(X_f_va)
            if cfg["target_mode"] == "log1p_l2":
                preds_va = np.expm1(preds_va)
            preds_va = np.clip(preds_va, 0.0, None)

            m_f = compute_count_metrics(y_f_va, preds_va)
            fold_rmsle.append(m_f["rmsle"])
            fold_mae.append(m_f["mae"])

        mean_rmsle = float(np.mean(fold_rmsle))
        mean_mae = float(np.mean(fold_mae))
        cv_results.append({**cfg, "cv_rmsle": round(mean_rmsle, 4), "cv_mae": round(mean_mae, 4)})

        with mlflow.start_run(run_name=f"trial_{idx:02d}", nested=True):
            mlflow.log_params(cfg)
            mlflow.log_metrics({"cv_rmsle": round(mean_rmsle, 4), "cv_mae": round(mean_mae, 4)})

cv_df = pd.DataFrame(cv_results).sort_values("cv_rmsle").reset_index(drop=True)
display(cv_df.head(8))

In [18]:
# Re-entrenar la mejor configuración en todo train_m y evaluar en Val (2024) y Test (2025)
best_cfg = cv_df.iloc[0].to_dict()
best_loss = "poisson" if best_cfg["target_mode"] == "raw_poisson" else "squared_error"
y_tr_best = train_m["casos"].to_numpy(dtype=float)
if best_cfg["target_mode"] == "log1p_l2":
    y_tr_best = np.log1p(y_tr_best)

best_hgb = HistGradientBoostingRegressor(
    loss=best_loss,
    learning_rate=float(best_cfg["learning_rate"]),
    max_iter=int(best_cfg["max_iter"]),
    max_leaf_nodes=int(best_cfg["max_leaf_nodes"]),
    min_samples_leaf=int(best_cfg["min_samples_leaf"]),
    l2_regularization=float(best_cfg["l2_regularization"]),
    categorical_features=cat_idx,
    random_state=42,
)
best_hgb.fit(train_m[hgb_cols], y_tr_best)

pred_va_best = best_hgb.predict(val_m[hgb_cols])
pred_te_best = best_hgb.predict(test_m[hgb_cols])
if best_cfg["target_mode"] == "log1p_l2":
    pred_va_best = np.expm1(pred_va_best)
    pred_te_best = np.expm1(pred_te_best)

val_m["pred_hgb_tuned"] = np.clip(pred_va_best, 0.0, None)
test_m["pred_hgb_tuned"] = np.clip(pred_te_best, 0.0, None)

val_hgb_t = compute_count_metrics(val_m["casos"], val_m["pred_hgb_tuned"], "val")
test_hgb_t = compute_count_metrics(test_m["casos"], test_m["pred_hgb_tuned"], "test")
print("HGB Tuned ->", val_hgb_t, test_hgb_t)

# Comparativa visual Auto-ARIMAX vs HGB Tuned
fig, axes = plt.subplots(2, 1, figsize=(13, 6.5), sharex=True)
eval_m = pd.concat([val_m, test_m], ignore_index=True)
for ax, cid, name in zip(axes, [101, 305], ["Cantón 101", "Cantón 305"]):
    sub = eval_m[eval_m["canton"] == cid].sort_values("time_idx")
    sub_sar = df[(df["canton"] == cid) & (df["year"] >= 2024)].sort_values("time_idx")
    ax.plot(sub["time_idx"], sub["casos"], color="black", lw=1.8, label="Casos reales")
    ax.plot(sub_sar["time_idx"], sub_sar["pred_sarimax"], ls="--", alpha=0.85, label="Auto-ARIMAX")
    ax.plot(sub["time_idx"], sub["pred_hgb_tuned"], alpha=0.9, label="HGB Tuned (TSCV)")
    ax.axvline(int(test_m["time_idx"].min()), color="gray", ls=":", label="Inicio Test (2025)")
    ax.set_title(f"Auto-ARIMAX vs HGB Tuned — {name}")
    ax.set_ylabel("Casos")
    ax.set_ylim(0, 300)
    ax.grid(False)
    ax.legend(loc="upper right", ncol=4)

fig.tight_layout()
fig_b2 = ROOT_DIR / "data/processed/eda/09_nivel1_hgb_tuned_vs_arimax.png"
fig.savefig(fig_b2, dpi=150)
plt.show()

En este caso se observa que Arimax tiene un mejor comprendimiento de los datos dados, es decir, en ambos casos con el modelo FineTunning se observa que no pudo llegar a su rendimiento.

In [14]:
with mlflow.start_run(run_name="01_hgb_tuned_best"):
    mlflow.set_tags({"stage": "nivel1_baseline", "horizon": "1_week", "split_policy": "train<=2023|val=2024|test=2025"})
    mlflow.log_params({k: v for k, v in best_cfg.items() if k not in ("cv_rmsle", "cv_mae")})
    mlflow.log_metrics({"cv_rmsle": best_cfg["cv_rmsle"], **val_hgb_t, **test_hgb_t})
    mlflow.log_artifact(str(fig_b2), artifact_path="nivel1_plots")

exp_runs = mlflow.search_runs(experiment_names=["dengue_cr_spatiotemporal"])
exp_runs = exp_runs[exp_runs["tags.stage"] != "nivel1_tuning"]
cols_show = [c for c in ["tags.mlflow.runName", "tags.stage", "metrics.val_mae", "metrics.val_rmsle", "metrics.val_wape", "metrics.test_mae", "metrics.test_rmsle", "metrics.test_wape"] if c in exp_runs.columns]
display(exp_runs[cols_show].sort_values("metrics.val_rmsle", na_position="last"))